<a href="https://colab.research.google.com/github/kite121/Machine-Learning-Course/blob/main/Lab9_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lab 9 Self practice

Objectives:
- understand the impact of number of layers in CNN, padding, strides, pooling (max vs average, etc.)

Let's laod the required libraries and packages

In [4]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

We will be working on the MNSIT dataset for our experiments.

In [5]:
# Model / data parameters
num_classes = 10
input_shape = (28, 28, 1)

# Load the data and split it between train and test sets
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Scale images to the [0, 1] range
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255
# Make sure images have shape (28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)
print("x_train shape:", x_train.shape)
print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")


# convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

x_train shape: (60000, 28, 28, 1)
60000 train samples
10000 test samples


Let's write a function that will create model based on the arguments that are passed into the function.
You have to write the the lines whose descriptions are given in the comments.

In [12]:
def create_model(
    number_of_layers=2,
    number_of_conv_kernels=[32,16],
    kernel_size_conv = [(3,3), (3,3)],
    padding_conv="valid",
    strides_conv=(1,1),
    pool_size=(2,2),
    is_pooling_average=False):

    if len(number_of_conv_kernels) != number_of_layers:
        raise ValueError('Number of elements in the number_of_conv_kernels should be equal to number_of_layers')
    if len(kernel_size_conv) != number_of_layers:
        raise ValueError('Number of elements in the number_of_conv_kernels should be equal to number_of_layers')

    model = keras.Sequential()
    model.add(keras.Input(shape=input_shape))

    for i in range(number_of_layers):

        # todo: write one line that:
        #   - creates a Conv2D layer; set the number of kernels;
        #   - sets the kernel size, strides, padding based on the values from the arguments of the function and activation function as 'relu';
        #   - adds the created layer into the model
        layer_1 = keras.layers.Conv2D(number_of_conv_kernels[i], kernel_size = kernel_size_conv[i], strides = strides_conv,
                                      padding = padding_conv, activation = "relu")
        model.add(layer_1)
        # todo: write 4 line that will create either AveragePooling2D or MaxPooling2D
        #   based on the is_pooling_average value passed into the function
        #   and sets the pool size based on the argument passed into the function
        if is_pooling_average:
          pooling = keras.layers.AveragePooling2D(pool_size= pool_size)
        else:
          pooling = keras.layers.MaxPooling2D(pool_size = pool_size)
        model.add(pooling)

    model.add(layers.Flatten())
    model.add(layers.Dense(num_classes, activation="softmax"))
    return model

We will write a function that will compile and train the passed model and return the accuracy

In [15]:
def train_model_and_get_accuracy(model):
    batch_size = 128
    epochs = 2

    # todo: write one line of code to compile the model with:
    #   - the categorical_crossentropy loss function
    #   - use adam as optimizer
    #   - use 'accuracy' as metrics

    model.compile(loss = "categorical_crossentropy", optimizer = "adamW", metrics = ["accuracy"])


    # todo: write one line of code to train the model:
    #   - set the size of the batch size
    #   - also, set train / validation dataset split as 90 / 10
    history = model.fit(x_train, y_train, batch_size = batch_size, epochs = epochs, validation_split = 0.1)

    score = model.evaluate(x_test, y_test, verbose=0)
    return score[1]

We can create several different combinations of parameters to try CNN models in different situations. Try to change the parameters that will increase the model accuracy.

In [16]:
import json

param1 = dict(number_of_layers=2,
    number_of_conv_kernels=[32,16],
    kernel_size_conv = [(3,3), (3,3)],
    padding_conv="valid",
    strides_conv=(1,1),
    pool_size=(2,2),
    is_pooling_average=False)

param2 = dict(number_of_layers=3,
    number_of_conv_kernels=[32,16,16],
    kernel_size_conv = [(3,3), (3,3), (3,3)],
    padding_conv="valid",
    strides_conv=(1,1),
    pool_size=(2,2),
    is_pooling_average=False)

param3 = dict(number_of_layers=2,
    number_of_conv_kernels=[32,16],
    kernel_size_conv = [(3,3), (3,3)],
    padding_conv="valid",
    strides_conv=(1,1),
    pool_size=(2,2),
    is_pooling_average=True)


param4 = dict(number_of_layers=2,
    number_of_conv_kernels=[64,32],
    kernel_size_conv = [(3,3), (3,3)],
    padding_conv="same",
    strides_conv=(1,1),
    pool_size=(3,3),
    is_pooling_average=False)

param_list = [param1, param2, param3, param4]

model_list = [create_model(**param) for param in param_list]

result_list = [train_model_and_get_accuracy(model) for model in model_list]

final_result = []


for i in range(len(param_list)):
    item = {
        'model arguments': param_list[i],
        'trainable param count': model_list[i].count_params(),
        'accuracy': result_list[i]
    }
    final_result.append(item)

final_result_json = {
    'items': final_result
}

print(json.dumps(final_result_json, indent = 4))

Epoch 1/2
422/422 ━━━━━━━━━━━━━━━━━━━━ 37s 83ms/step - accuracy: 0.8815 - loss: 0.4125 - val_accuracy: 0.9683 - val_loss: 0.1181
Epoch 2/2
422/422 ━━━━━━━━━━━━━━━━━━━━ 31s 74ms/step - accuracy: 0.9670 - loss: 0.1091 - val_accuracy: 0.9757 - val_loss: 0.0846
Epoch 1/2
422/422 ━━━━━━━━━━━━━━━━━━━━ 35s 78ms/step - accuracy: 0.7946 - loss: 0.6589 - val_accuracy: 0.9238 - val_loss: 0.2476
Epoch 2/2
422/422 ━━━━━━━━━━━━━━━━━━━━ 33s 79ms/step - accuracy: 0.9330 - loss: 0.2160 - val_accuracy: 0.9525 - val_loss: 0.1724
Epoch 1/2
422/422 ━━━━━━━━━━━━━━━━━━━━ 29s 66ms/step - accuracy: 0.8359 - loss: 0.5644 - val_accuracy: 0.9397 - val_loss: 0.2103
Epoch 2/2
422/422 ━━━━━━━━━━━━━━━━━━━━ 41s 66ms/step - accuracy: 0.9433 - loss: 0.1960 - val_accuracy: 0.9648 - val_loss: 0.1290
Epoch 1/2
422/422 ━━━━━━━━━━━━━━━━━━━━ 59s 134ms/step - accuracy: 0.8766 - loss: 0.4362 - val_accuracy: 0.9682 - val_loss: 0.1086
Epoch 2/2
422/422 ━━━━━━━━━━━━━━━━━━━━ 57s 134ms/step - accuracy: 0.9662 - loss: 0.1084 - val_ac